# driver_DD_NMROM
Driver to implement and test NM ROM on the 2D Burgers Equation.  

In [ ]:
import sys
with open("./../../../PATHS.txt") as file:
  paths = file.read().splitlines()
sys.path.extend(paths)

In [ ]:
from dd_nm_rom import env
env.set(
  backend="numpy",
  device="cpu",
  device_idx=0,
  nb_threads=4,
  epsilon=1e-10,
  floatx="float64",
  seed=0
)

In [ ]:
import os
import numpy as np
import dill as pickle
import matplotlib.pyplot as plt
import dill as pickle

from matplotlib import cm

In [ ]:
from dd_nm_rom import ops, utils
from dd_nm_rom import fom as fom_mod
from dd_nm_rom import rom as rom_mod
from dd_nm_rom import field as field_mod
from dd_nm_rom.elements import mesh as mesh_mod

In [ ]:
prefix = "/usr/workspace/zanardi1/Codes/DD-NM-ROM/run_old2/steady/"
paths = {
  "pod": prefix + "/dset.02/pod/run.01/residuals/",
  "data": prefix + "/dset.02/datagen/",
  "nets": prefix + "/dset.01/nets/run.01/",
  "figs": prefix + "/dset.01/figures/"
}
for k in ("figs",):
  os.makedirs(paths[k], exist_ok=True)

In [ ]:
plt.rc('font', size=20)
plt.rcParams['text.usetex'] = False

## Set model parameters

In [ ]:
# define constant parameters for PDE
nx, ny  = 480, 24
x_lim   = [-1.0, 1.0]
y_lim   = [0.0, 0.05]

na1, nlam = 80, 80
a_lim  = [1.0, 10000.0]
k_lim = [5.0, 25.0]

a1, lam = 7692.5384, 21.9230
mu = np.array([a1, lam])

viscosity = 1e-1

# number of subdomains in x and y directions for DD model
n_sub_x = 2
n_sub_y = 2

scaling = -1 # scaling factor for residual. -1 uses hx*hy

In [ ]:
Mu = pickle.load(open(paths["data"]+"/mu.p", "rb"))

In [ ]:
residuals = utils.load_case_parallel(
  n_workers=32,
  ranges=[0,400],
  key="residuals",
  path=paths["data"]
)
residuals = np.vstack([x for x in residuals if x is not None])

In [ ]:
snapshots = utils.load_case_parallel(
  n_workers=32,
  ranges=[0,400],
  key="snapshots",
  path=paths["data"]
)
snapshots = np.vstack([x for x in snapshots if x is not None])

## Solve DD FOM

In [ ]:
mesh_mono = mesh_mod.MeshMono(
  nx=nx,
  ny=ny,
  x_lim=x_lim,
  y_lim=y_lim
)
mesh_mono.build()
mesh = mesh_mod.MeshDD(
  **mesh_mono.get_config_dd(n_sub_x, n_sub_y)
)
mesh.build()
X, Y = mesh.grid

In [ ]:
field = field_mod.Burgers2DExact(
  mesh=mesh,
  nu=viscosity,
  a_lim=a_lim,
  k_lim=k_lim
)
field.set_params(mu)
fom = fom_mod.Burgers2D(
  nu=viscosity,
  mesh=mesh
)
fom.build(field)
dd_fom = fom_mod.DDBurgers2D(fom, constraint_type='strong', scaling=scaling)
dd_fom.build()

In [ ]:
print('\nSolving full domain model:')
# generate Burgers FOM
uv, rhs, converged = fom.solve(tol=1e-8, maxit=20, stepsize_min=1e-20, verbose=True)
sol = np.concatenate([uv["u"], uv["v"]])
print("RUNTIME:", fom.runtime)

print('\nSolving DD model:')
# compute DD model
uv_dd, lambdas, rhs_dd, converged = dd_fom.solve(tol=1e-8, maxit=50, stepsize_min=1e-20, verbose=True)
sol_dd = np.concatenate([uv_dd["res"]["u"], uv_dd["res"]["v"]])
print("RUNTIME:", dd_fom.runtime)

# dd_fom_rel_err = np.linalg.norm(sol_dd-sol)/np.linalg.norm(sol)
dd_fom_rel_err = 100*np.mean(np.abs(sol_dd-sol)/(np.abs(sol)+1e-5))
print(f'\nDD-FOM mean relative error = {dd_fom_rel_err:1.4e} %') 

In [ ]:
U_fom = uv["u"].reshape(ny, nx)
V_fom = uv["v"].reshape(ny, nx)

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, U_fom, cmap=cm.jet, shading='auto', vmin=uv["u"].min(), vmax=uv["u"].max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$')
plt.tight_layout()
# file = paths["figs"]+'/u_fom.png'
# plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, V_fom, cmap=cm.jet, shading='auto', vmin=uv["v"].min(), vmax=uv["v"].max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$')
plt.tight_layout()
# file = paths["figs"]+'/v_fom.png'
# plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
U_dd_fom = uv_dd["res"]["u"].reshape(ny, nx)
V_dd_fom = uv_dd["res"]["v"].reshape(ny, nx)

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, U_dd_fom, cmap=cm.jet, shading='auto', vmin=uv["u"].min(), vmax=uv["u"].max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$')
plt.tight_layout()
file = paths["figs"]+'/u_fom.png'
plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, V_dd_fom, cmap=cm.jet, shading='auto', vmin=uv["v"].min(), vmax=uv["v"].max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$')
plt.tight_layout()
file = paths["figs"]+'/v_fom.png'
plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
U_err = np.abs(U_dd_fom-U_fom)/np.linalg.norm(U_fom) #(np.abs(U_fom)+1e-5)
V_err = np.abs(V_dd_fom-V_fom)/np.linalg.norm(V_fom) #(np.abs(V_fom)+1e-5)

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, U_err, cmap=cm.jet, shading='auto')#, vmin=uv["u"].min(), vmax=uv["u"].max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$')
plt.tight_layout()
# file = paths["figs"]+'/u_fom.png'
# plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, V_err, cmap=cm.jet, shading='auto')#, vmin=uv["v"].min(), vmax=uv["v"].max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$')
plt.tight_layout()
# file = paths["figs"]+'/v_fom.png'
# plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

## Solve DD NM-ROM

Load SVD data

In [ ]:
print('Loading POD data...')
pod_bases = pickle.load(open(paths["pod"]+'/multi/bases.p', 'rb'))
res_bases = pod_bases["res"]
print('Data loaded!')

print('\nResidual bases:')
print(ops.map_nested_dict(res_bases, np.shape))

Define paths to NN models

In [ ]:
single_nn_model = True

In [ ]:
if single_nn_model:
  path_to_nets = paths["nets"] + "/merged/"
else:
  path_to_nets = paths["nets"] + "/multi/"

nn_configfiles = {
  "port": [],
  "interior": [],
  "interface": []
}

# Ports
for i in range(len(dd_fom.dd_indices.ports)):
  size = dd_fom.dd_indices.port_to_nodes[i].size
  for file in os.scandir(path_to_nets + "/ports/"):
    if ((file.is_dir()) and ("ports" in file.name)):
      if ((str(size) in file.name) and (str(i) in file.name)):
        if (size == 1):
          filename = file.path + "/saving/model_last_numpy.p"
        else:
          filename = file.path + "/training/ckpt/model_best_numpy.p"
        nn_configfiles["port"].append(filename)

# Interior
for i in range(mesh.n_sub):
  for file in os.scandir(path_to_nets + "/interior/"):
    if ((file.is_dir()) and ("subs" in file.name)):
      if (str(i) in file.name):
        filename = file.path + "/training/ckpt/model_best_numpy.p"
        nn_configfiles["interior"].append(filename)

# Interface
for i in range(mesh.n_sub):
  for file in os.scandir(path_to_nets + "/interface/"):
    if ((file.is_dir()) and ("subs" in file.name)):
      if (str(i) in file.name):
        filename = file.path + "/training/ckpt/model_best_numpy.p"
        nn_configfiles["interface"].append(filename)

In [ ]:
# build ROM and RBF model (used for generating initial iterate)
print('Building DD NM-ROM...')
ddnmrom = rom_mod.DD_NM_ROM(
  dd_fom=dd_fom,
  nn_configfiles=nn_configfiles,
  res_bases=res_bases,
  hr_active=False,
  hr_n_samples=100,
  hr_n_edge_samples_ratio=0.75,
  hr_sample_small_ports=True,
  hr_small_ports_dim=5,
  constraint_type="weak",
  n_constraints_weak=8,
  scaling=scaling
)
print('ROM built!\n')

In [ ]:
print('Computing RBF interpolant ...')
dd_data = dd_fom.map_sol_on_elements(np.vstack(snapshots), map_on_ports=False)
ddnmrom.rbf_model.build(dd_data, Mu, smoothing=0.0, kernel='linear')
print('Interpolant computed!')

In [ ]:
# solve DD ROM
print('Solving DD NM-ROM...')
uv_dd_rom, z_rom, lambdas, rhs, converged = ddnmrom.solve(mu=np.array([a1, lam]), tol=1e-8, maxit=500, stepsize_min=1e-10, verbose=True)
print('Solution found!\n')
print("RUNTIME:", ddnmrom.runtime)

# compute error
dd_rom_rel_err = ddnmrom.compute_error(uv_dd, uv_dd_rom)
print(f'DD NM-ROM rel. error = {dd_rom_rel_err:1.4e}')

Plot DD ROM u and v

In [ ]:
nm_figs = paths["figs"] + '/nmrom/'
nm_figs += '/single/' if single_nn_model else '/multi/'
os.makedirs(nm_figs, exist_ok=True)

In [ ]:
# plot DD ROM u and v
filename = nm_figs
filename += '/srpc' if ddnmrom.constraint_type == 'strong' else '/wfpc'
filename += '_col_hr' if ddnmrom.hr_active else ''

# plot DD FOM u and v
x = np.linspace(x_lim[0], x_lim[1], nx+2)[1:-1]
y = np.linspace(y_lim[0], y_lim[1], ny+2)[1:-1]
X, Y = np.meshgrid(x, y)
U_rom = uv_dd_rom["res"]["u"].reshape(ny, nx)
V_rom = uv_dd_rom["res"]["v"].reshape(ny, nx)

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, U_rom, cmap=cm.jet, shading='auto', vmin=U_fom.min(), vmax=U_fom.max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$\hat{u}$')
plt.tight_layout()
file = filename+'_u.png'
plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure(figsize=(12, 4))
plt.pcolormesh(X, Y, V_rom, cmap=cm.jet, shading='auto', vmin=V_fom.min(), vmax=V_fom.max())
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$\hat{v}$')
plt.tight_layout()
file = filename+'_v.png'
plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

In [ ]:
# plot errors
u_rel_err = np.abs(U_fom-U_rom) / np.linalg.norm(U_fom)
plt.figure(figsize=(12,4))
plt.pcolormesh(X, Y, u_rel_err, cmap=cm.jet, shading='auto')#, vmin=0.0, vmax=4.88e-4)
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$u$ error', format='%.0e')
plt.tight_layout()
file = filename+'_u_error.png'
plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()

plt.figure(figsize=(12,4))
v_rel_err = np.abs(V_fom-V_rom)/np.linalg.norm(V_fom)
plt.pcolormesh(X, Y, v_rel_err, cmap=cm.jet, shading='auto')#, vmin=0.0, vmax=4.88e-4)
plt.xlabel('$x$')
plt.ylabel('$y$')
cb = plt.colorbar(label='$v$ error', format='%.0e')
plt.tight_layout()
file = filename+'_v_error.png'
plt.savefig(file, bbox_inches='tight', pad_inches=0.1)
plt.show()